# Práctica: Mecanismo de atención y Transformer básico sobre una serie de tiempo de temperatura
Dataset: temperaturas mínimas diarias en Melbourne (1981-1990), 3650 observaciones.
Tarea: dada una ventana de 30 días, predecir la temperatura mínima del día siguiente.

In [ ]:
import json, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(1)
URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"
df = pd.read_csv(URL)
df["Date"] = pd.to_datetime(df["Date"])
serie = df["Temp"].astype(float).values
print("Observaciones:", len(serie), "| min/max/media:", serie.min(), serie.max(), round(serie.mean(), 2))

## 1. Preparación de datos
División cronológica 70/15/15, normalización con estadísticas del entrenamiento y ventanas deslizantes.

In [ ]:
VENT = 30
n = len(serie)
i_tr, i_va = int(n * 0.70), int(n * 0.85)
mu, sd = serie[:i_tr].mean(), serie[:i_tr].std()
z = (serie - mu) / sd

def ventanas(z, ini, fin, v=VENT):
    X, y = [], []
    for t in range(max(ini, v), fin):
        X.append(z[t - v:t]); y.append(z[t])
    return (torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1),
            torch.tensor(np.array(y), dtype=torch.float32))

Xtr, ytr = ventanas(z, 0, i_tr)
Xva, yva = ventanas(z, i_tr, i_va)
Xte, yte = ventanas(z, i_va, n)
print("train/val/test:", len(Xtr), len(Xva), len(Xte))

## 2. Modelos
**Modelo A:** atención simple (una cabeza, Q/K/V) sobre la ventana + capa lineal.
**Modelo B:** Transformer básico (encoder): proyección, codificación posicional, 2 bloques con
atención multi-cabeza, red feed-forward, residuales y LayerNorm, y capa de salida.

In [ ]:
class PosEnc(nn.Module):
    def __init__(self, d, maxlen=200):
        super().__init__()
        pe = torch.zeros(maxlen, d)
        pos = torch.arange(maxlen).unsqueeze(1)
        div = torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class AtencionSimple(nn.Module):
    """Atención escalada de producto punto: softmax(QK^T/sqrt(d)) V, con la consulta del último paso."""
    def __init__(self, d=32):
        super().__init__()
        self.inp = nn.Linear(1, d); self.pos = PosEnc(d)
        self.q = nn.Linear(d, d); self.k = nn.Linear(d, d); self.v = nn.Linear(d, d)
        self.out = nn.Linear(d, 1); self.d = d
        self.pesos = None
    def forward(self, x):
        h = self.pos(self.inp(x))
        q = self.q(h[:, -1:, :])                       # consulta: último día
        k, v = self.k(h), self.v(h)
        a = torch.softmax(q @ k.transpose(1, 2) / math.sqrt(self.d), dim=-1)
        self.pesos = a.detach()
        ctx = a @ v
        return self.out(ctx.squeeze(1)).squeeze(-1)

class BloqueEncoder(nn.Module):
    def __init__(self, d, h, dff, p=0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(d, h, dropout=p, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, dff), nn.ReLU(), nn.Dropout(p), nn.Linear(dff, d))
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.dp = nn.Dropout(p)
        self.pesos = None
    def forward(self, x):
        a, w = self.mha(x, x, x, need_weights=True, average_attn_weights=True)
        self.pesos = w.detach()
        x = self.n1(x + self.dp(a))
        return self.n2(x + self.dp(self.ff(x)))

class TransformerBasico(nn.Module):
    def __init__(self, d=32, h=4, dff=64, capas=2, p=0.1):
        super().__init__()
        self.inp = nn.Linear(1, d); self.pos = PosEnc(d)
        self.bloques = nn.ModuleList([BloqueEncoder(d, h, dff, p) for _ in range(capas)])
        self.out = nn.Linear(d, 1)
    def forward(self, x):
        h = self.pos(self.inp(x))
        for b in self.bloques:
            h = b(h)
        return self.out(h[:, -1, :]).squeeze(-1)    # representación del último paso

def entrenar(modelo, epocas=60, lr=2e-3, bs=64, paciencia=8):
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    fn = nn.MSELoss()
    mejor, mejor_estado, sin_mejora = 1e9, None, 0
    hist = {"train": [], "val": []}
    for ep in range(epocas):
        modelo.train()
        perm = torch.randperm(len(Xtr))
        tot = 0.0
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]
            opt.zero_grad()
            loss = fn(modelo(Xtr[idx]), ytr[idx])
            loss.backward(); opt.step()
            tot += loss.item() * len(idx)
        modelo.eval()
        with torch.no_grad():
            lv = fn(modelo(Xva), yva).item()
        hist["train"].append(tot / len(Xtr)); hist["val"].append(lv)
        if lv < mejor - 1e-5:
            mejor, sin_mejora = lv, 0
            mejor_estado = {k: v.clone() for k, v in modelo.state_dict().items()}
        else:
            sin_mejora += 1
            if sin_mejora >= paciencia:
                break
    modelo.load_state_dict(mejor_estado)
    return hist

def evaluar(pred_z, y_z):
    p, r = pred_z * sd + mu, y_z * sd + mu
    err = p - r
    return {"MAE": float(np.mean(np.abs(err))), "RMSE": float(np.sqrt(np.mean(err ** 2))),
            "R2": float(1 - np.sum(err ** 2) / np.sum((r - r.mean()) ** 2))}

## 3. Entrenamiento y evaluación

In [ ]:
res, hists, preds = {}, {}, {}
yte_np = yte.numpy()

# Línea base: persistencia (el valor de ayer) y media móvil de 7 días
pers = Xte[:, -1, 0].numpy()
res["Persistencia (ayer)"] = evaluar(pers, yte_np)
mm7 = Xte[:, -7:, 0].numpy().mean(axis=1)
res["Media móvil 7 días"] = evaluar(mm7, yte_np)

torch.manual_seed(SEED)
mA = AtencionSimple(); t0 = time.time()
hists["Atención simple"] = entrenar(mA); tA = time.time() - t0
mA.eval()
with torch.no_grad():
    pA = mA(Xte).numpy()
res["Atención simple"] = evaluar(pA, yte_np); preds["Atención simple"] = pA
pesosA = mA.pesos.squeeze(1).mean(0).numpy()

torch.manual_seed(SEED)
mB = TransformerBasico(); t0 = time.time()
hists["Transformer básico"] = entrenar(mB); tB = time.time() - t0
mB.eval()
with torch.no_grad():
    pB = mB(Xte).numpy()
res["Transformer básico"] = evaluar(pB, yte_np); preds["Transformer básico"] = pB
pesosB = mB.bloques[-1].pesos[:, -1, :].mean(0).numpy()   # atención del último paso, promedio en test

params = {"Atención simple": sum(p.numel() for p in mA.parameters()),
          "Transformer básico": sum(p.numel() for p in mB.parameters())}
for k, v in res.items():
    print(f"{k:22s} MAE={v['MAE']:.3f}  RMSE={v['RMSE']:.3f}  R2={v['R2']:.3f}")
print("Parámetros:", params, "| épocas:", {k: len(h['val']) for k, h in hists.items()},
      "| tiempo(s):", round(tA, 1), round(tB, 1))

## 4. Gráficas

In [ ]:
plt.rcParams.update({"font.size": 10})
fig, ax = plt.subplots(figsize=(7, 3.4))
for k, h in hists.items():
    ax.plot(h["train"], label=f"{k} (entrenamiento)")
    ax.plot(h["val"], "--", label=f"{k} (validación)")
ax.set_xlabel("Época"); ax.set_ylabel("Pérdida MSE (datos normalizados)")
ax.set_title("Curvas de aprendizaje"); ax.legend(fontsize=7); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig("fig_perdida.png", dpi=200); plt.show()

fechas = df["Date"].values[i_va:][:len(yte_np)]
N = 120
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(fechas[:N], yte_np[:N] * sd + mu, "k-", lw=1.6, label="Real")
ax.plot(fechas[:N], pA[:N] * sd + mu, "-", lw=1.1, label="Atención simple")
ax.plot(fechas[:N], pB[:N] * sd + mu, "-", lw=1.1, label="Transformer básico")
ax.set_ylabel("Temperatura mínima (°C)"); ax.set_title("Predicción vs. valor real (primeros 120 días de prueba)")
ax.legend(fontsize=8); ax.grid(alpha=.3); fig.autofmt_xdate()
fig.tight_layout(); fig.savefig("fig_prediccion.png", dpi=200); plt.show()

fig, axs = plt.subplots(1, 2, figsize=(7, 3.0), sharey=False)
dias = np.arange(-VENT, 0)
axs[0].bar(dias, pesosA, color="#2b6cb0"); axs[0].set_title("Atención simple", fontsize=10)
axs[1].bar(dias, pesosB, color="#c05621"); axs[1].set_title("Transformer (última capa)", fontsize=10)
for a in axs:
    a.set_xlabel("Día relativo en la ventana"); a.grid(alpha=.3)
axs[0].set_ylabel("Peso de atención promedio")
fig.tight_layout(); fig.savefig("fig_atencion.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.plot(df["Date"], serie, lw=.5, color="#444")
ax.axvline(df["Date"].iloc[i_tr], color="r", ls="--", lw=1); ax.axvline(df["Date"].iloc[i_va], color="r", ls="--", lw=1)
ax.set_ylabel("°C"); ax.set_title("Serie completa y división train / val / test")
ax.grid(alpha=.3); fig.tight_layout(); fig.savefig("fig_serie.png", dpi=200); plt.show()

json.dump({"res": res, "params": params,
           "epocas": {k: len(h["val"]) for k, h in hists.items()},
           "tiempo": {"A": tA, "B": tB},
           "split": [len(Xtr), len(Xva), len(Xte)],
           "mu": float(mu), "sd": float(sd),
           "pesosA_top": [int(i) for i in np.argsort(-pesosA)[:3] - VENT],
           "pesosB_top": [int(i) for i in np.argsort(-pesosB)[:3] - VENT],
           "pesosA_ultimo": float(pesosA[-1]), "pesosB_ultimo": float(pesosB[-1]),
           "pesosA_max": float(pesosA.max()), "pesosB_max": float(pesosB.max())},
          open("resultados.json", "w"), indent=1)